# Marker — PDF → Markdown (local, ML pipeline, Apple Silicon)

Experiment with [**Marker**](https://github.com/datalab-to/marker) (`marker-pdf`, by Datalab —
the [Surya](https://github.com/datalab-to/surya) authors): a deep-learning document pipeline that
turns **PDF / image / DOCX / PPTX / XLSX / EPub / HTML** into **Markdown / JSON / HTML**, running
**layout detection → OCR → table recognition → equation (texify) → reading order**. It extracts
figures, renders tables, and converts equations to **LaTeX**.

Marker is the **other heavyweight** alongside MinerU — a real OCR/layout stack (not a text-layer
reader), so it handles **scanned / image-only PDFs**. Its pipeline runs on CPU and
**Apple-Silicon (MPS)**, so this notebook runs **locally on your Mac**.

Where it sits among the four notebooks in this repo:

| approach | needs | speed | scanned PDFs | equations |
|---|---|---|---|---|
| **Marker (this notebook)** | Surya models, ~1–2 GB, MPS/CPU | ~sec–min/doc | ✅ OCRs | ✅ LaTeX (texify) |
| MinerU (`../mineru`) | layout+OCR+table+formula models | ~15–240 s/doc | ✅ OCRs | ✅ LaTeX |
| PyMuPDF4LLM (`../pymupdf4llm`) | pure Python | ~0.1–2 s/doc | ❌ text layer only | ❌ |
| MarkItDown (`../markitdown`) | pure Python (pdfminer) | ~0.05–1 s/doc | ❌ text layer only | ❌ |

**Optional LLM boost** (not wired here — this run stays offline): `config={'use_llm': True}` plus
an LLM service improves tables / equations / inline math. This notebook runs the **base pipeline
only**. The manifest columns match the other three notebooks so the four `*_manifest.csv` files
diff 1:1.

## 0. Environment (read first — one-time setup)

Marker needs **Python 3.10–3.13** and pulls **PyTorch**. We use a dedicated 3.12 venv. From the
repo root, in a terminal:

```bash
# 1. a modern Python (Homebrew)
brew install python@3.12

# 2. a venv just for Marker
/opt/homebrew/bin/python3.12 -m venv .venv-marker
source .venv-marker/bin/activate

# 3. Marker (pulls torch) + pypdf (page counts)
python -m pip install --upgrade pip
pip install -U marker-pdf pypdf

# 4. make this venv a Jupyter kernel, then launch
pip install jupyter ipykernel pandas markdown
python -m ipykernel install --user --name marker --display-name "Python (marker)"
jupyter lab   # or: jupyter notebook
```

Then open this notebook and pick the **Python (marker)** kernel (top-right kernel picker).


The **first conversion downloads the Surya models** (~1–2 GB: layout, recognition/OCR, table,
texify) into `~/.cache`. One time only; later runs are offline.

## 1. Sanity check — right Python, Marker importable, device

In [1]:
import sys, os, platform, importlib.metadata as _m

print('Python :', sys.version.split()[0], '(need 3.10-3.13)')
assert (3, 10) <= sys.version_info[:2] <= (3, 13), (
    'Select the "Python (marker)" kernel — see section 0.'
)

# Device: Apple-Silicon -> mps, NVIDIA -> cuda, else cpu. Set BEFORE marker/torch load.
import torch
def detect_device():
    if torch.cuda.is_available():
        return 'cuda'
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'
DEVICE = detect_device()
os.environ.setdefault('TORCH_DEVICE', DEVICE)

print('marker-pdf:', _m.version('marker-pdf'))
print('torch     :', torch.__version__)
print('machine   :', platform.platform())
print('device    :', DEVICE, '(TORCH_DEVICE=%s)' % os.environ['TORCH_DEVICE'])

Python : 3.12.13 (need 3.10-3.13)
marker-pdf: 1.10.2
torch     : 2.13.0
machine   : macOS-26.5.2-arm64-arm-64bit
device    : mps (TORCH_DEVICE=mps)


## 2. Config

Everything is relative to this notebook's folder (`notebooks/marker/`), so the PDFs resolve to
`ocr_exploration/pdf_resources/`. Defaults mirror the other runs so the manifests line up.

In [2]:
from pathlib import Path

HERE      = Path.cwd()
# Robust to being launched from repo root or from notebooks/marker/.
REPO_ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'pdf_resources').is_dir()), HERE)
PDF_DIR   = REPO_ROOT / 'pdf_resources'
OUT_DIR   = HERE / 'out'
HTML_DIR  = HERE / 'html'

# The Atlanta Code of Ordinances is hundreds of pages; cap to match the other runs.
MAX_PAGES = 15            # None -> every page of every PDF
MAX_FILES = None          # e.g. 2 -> smoke-test the first 2 PDFs; None -> all
FORCE_OCR = False         # True -> re-OCR every page (ignore embedded text layer)
USE_LLM   = False         # True needs an LLM service + API key; left off for an offline run

OUT_DIR.mkdir(exist_ok=True)
HTML_DIR.mkdir(exist_ok=True)
print('PDF_DIR :', PDF_DIR)
print('OUT_DIR :', OUT_DIR)
print('max_pages/max_files/force_ocr/use_llm:', MAX_PAGES, MAX_FILES, FORCE_OCR, USE_LLM)

PDF_DIR : /Users/asikifthakerhamim/Documents/ocr_exploration/pdf_resources
OUT_DIR : /Users/asikifthakerhamim/Documents/ocr_exploration/notebooks/marker/out
max_pages/max_files/force_ocr/use_llm: 15 None False False


## 3. Discover the PDFs

In [3]:
import pypdf

pdfs = sorted(PDF_DIR.glob('*.pdf'))
if MAX_FILES:
    pdfs = pdfs[:MAX_FILES]
assert pdfs, f'No PDFs under {PDF_DIR}'

for p in pdfs:
    print(f'{p.stat().st_size/1024:8.0f} KB  {p.name}')
print(f'\n{len(pdfs)} PDF(s)')

     787 KB  AF SE 2.pdf
     206 KB  Admin Eval Framework.docx.pdf
     666 KB  Agenda_ Operational Leadership Institute (SY 26-27).pdf
     274 KB  Athlos 2024-2025 Observation Rubric Teacher.docx.pdf
    2076 KB  Atlanta, GA Code of Ordinances.pdf
     224 KB  Benefits_HR Specialist Evaluation Form Template  (1).pdf
     224 KB  Benefits_HR Specialist Evaluation Form Template .pdf
      81 KB  Bullseye Math Checklist  - Sheet1.pdf
    2431 KB  Classroom Environment Checklist 6-8.docx.pdf
     130 KB  Marzano Rubrics with scales. potential evidence, and elements 2022.pdf

10 PDF(s)


## 4. Load the Surya models (once)

`create_model_dict()` loads the layout, recognition/OCR, table, and texify models onto `DEVICE`.
The **first run downloads them** (~1–2 GB) — watch the progress bars. We build **one**
`PdfConverter` and reuse it for every PDF; the `page_range` is set per file in the next cell.

In [5]:
import time
from marker.models import create_model_dict

t0 = time.perf_counter()
ARTIFACTS = create_model_dict()
print(f'models loaded on {DEVICE} in {time.perf_counter()-t0:.1f}s')

2026-07-20 17:49:57,322 [WARNING] surya: `TableRecEncoderDecoderModel` is not compatible with mps backend. Defaulting to cpu instead


models loaded on mps in 1.9s


## 5. Convert each PDF with Marker

For each PDF we build a fresh `PdfConverter` with a per-file `config` (`page_range` for the page
cap, `force_ocr`, `use_llm`), convert, then pull `(markdown, ext, images)` out via
`text_from_rendered`. We write `out/<safe>/<safe>.md`, the extracted figures to `images/`, and the
Marker metadata (per-page block counts + OCR method) to `<safe>.meta.json`. Folder names are
sanitized so spaces/parens don't bite downstream tools.

In [6]:
import json
from marker.converters.pdf import PdfConverter
from marker.output import text_from_rendered

def safe_name(stem: str) -> str:
    for a, b in [('(', '-'), (')', '-'), ('[', '-'), (']', '-'), (' ', '_')]:
        stem = stem.replace(a, b)
    for dash in '\u2010\u2011\u2012\u2013\u2014\u2015\u2212':
        stem = stem.replace(dash, '-')
    return stem

def convert(pdf: Path) -> dict:
    stem = pdf.stem
    safe = safe_name(stem)
    out  = OUT_DIR / safe
    img_dir = out / 'images'
    img_dir.mkdir(parents=True, exist_ok=True)

    total = len(pypdf.PdfReader(str(pdf)).pages)
    cfg = {'force_ocr': FORCE_OCR, 'use_llm': USE_LLM,
           'output_format': 'markdown'}
    if MAX_PAGES is not None:
        cfg['page_range'] = list(range(min(MAX_PAGES, total)))

    t0 = time.perf_counter()
    try:
        converter = PdfConverter(artifact_dict=ARTIFACTS, config=cfg)
        rendered = converter(str(pdf))
        text, _ext, images = text_from_rendered(rendered)
        dt = time.perf_counter() - t0
        ok, err = True, ''
    except Exception as e:
        dt = time.perf_counter() - t0
        ok, err, text, images, rendered = False, repr(e), '', {}, None
        print(f'  !! {stem} failed: {err}')

    md_path, meta = '', {}
    if ok:
        # save extracted figures, rewrite refs to portable images/<name>
        for name, img in images.items():
            try:
                img.save(img_dir / name)
            except Exception:
                pass
        (out / f'{safe}.md').write_text(text, encoding='utf-8')
        md_path = str(out / f'{safe}.md')
        meta = getattr(rendered, 'metadata', {}) or {}
        (out / f'{safe}.meta.json').write_text(
            json.dumps(meta, ensure_ascii=False, default=str, indent=1), encoding='utf-8')

    used = (min(MAX_PAGES, total) if MAX_PAGES is not None else total)
    return {'stem': stem, 'safe': safe, 'ok': ok, 'latency_s': round(dt, 2),
            'pages': used, 'total_pages': total, 'n_img_files': len(images),
            'md_path': md_path, 'meta': meta, 'err': err}

runs = []
for i, pdf in enumerate(pdfs, 1):
    print(f'[{i}/{len(pdfs)}] {pdf.name} ...', flush=True)
    r = convert(pdf)
    if r['ok']:
        cap = f' (of {r["total_pages"]})' if r['pages'] != r['total_pages'] else ''
        print(f'      done in {r["latency_s"]}s — {r["pages"]} page(s){cap}, {r["n_img_files"]} image(s)')
    runs.append(r)
print('\nconversion pass complete')

[1/10] AF SE 2.pdf ...


Recognizing Text: 100%|██████████| 30/30 [00:44<00:00,  1.49s/it]

      done in 116.61s — 15 page(s) (of 37), 9 image(s)
[2/10] Admin Eval Framework.docx.pdf ...



Running OCR Error Detection: 100%|██████████| 2/2 [00:00<00:00, 10.11it/s]
Detecting bboxes: 0it [00:00, ?it/s]
Detecting bboxes: 0it [00:00, ?it/s]

      done in 22.23s — 7 page(s), 0 image(s)
[3/10] Agenda_ Operational Leadership Institute (SY 26-27).pdf ...


...

      done in 1034.27s — 15 page(s) (of 33), 0 image(s)
[9/10] Classroom Environment Checklist 6-8.docx.pdf ...


Recognizing tables: 100%|██████████| 1/1 [00:04<00:00,  4.97s/it]
Detecting bboxes: 0it [00:00, ?it/s]


      done in 26.52s — 5 page(s), 3 image(s)
[10/10] Marzano Rubrics with scales. potential evidence, and elements 2022.pdf ...


Running OCR Error Detection: 100%|██████████| 3/3 [00:00<00:00, 19.77it/s]
Detecting bboxes: 0it [00:00, ?it/s]
Recognizing tables: 100%|██████████| 3/3 [00:09<00:00,  3.11s/it]
Detecting bboxes: 0it [00:00, ?it/s]

      done in 39.07s — 10 page(s), 0 image(s)

conversion pass complete


## 6. Collect outputs & compute per-PDF stats

Marker's metadata carries **`page_stats`** — per page, a `block_counts` list of `[block_type,
count]`. We aggregate those into the **same columns the other manifests use**: `Table` → tables,
`Picture`/`Figure` → images, `Equation` → equations, `Text` → text blocks. If `page_stats` is
absent we fall back to parsing the Markdown (GFM tables, `![]()` refs, `$$` math).

In [7]:
import re
from collections import Counter

TABLE_SEP = re.compile(r'(?m)^\s*\|(?:\s*:?-{3,}:?\s*\|)+\s*$')
IMG_REF   = re.compile(r'!\[[^\]]*\]\([^)]*\)')
DISP_MATH = re.compile(r'(?ms)(?<!\$)\$\$(?!\$).+?\$\$')

def block_counts(meta):
    kinds = Counter()
    for pg in (meta or {}).get('page_stats', []) or []:
        for item in pg.get('block_counts', []) or []:
            try:
                name, n = item
            except Exception:
                continue
            kinds[name] += int(n)
    return kinds

def stats_for(meta, text, n_img_files):
    kinds = block_counts(meta)
    if kinds:  # authoritative: Marker's own block tally
        return (kinds.get('Table', 0),
                kinds.get('Picture', 0) + kinds.get('Figure', 0),
                kinds.get('Equation', 0),
                kinds.get('Text', 0))
    # fallback from the markdown
    return (len(TABLE_SEP.findall(text)),
            n_img_files or len(IMG_REF.findall(text)),
            len(DISP_MATH.findall(text)),
            sum(1 for b in re.split(r'\n\s*\n', text) if b.strip()))

records = []
for r in runs:
    text = Path(r['md_path']).read_text(encoding='utf-8') if r['md_path'] else ''
    n_tables, n_images, n_equations, n_text = stats_for(r['meta'], text, r['n_img_files'])
    records.append({
        'pdf': r['stem'],
        'ok': r['ok'],
        'backend': 'marker',
        'latency_s': r['latency_s'],
        'pages': r['pages'],
        'chars': len(text),
        'n_tables': n_tables,
        'n_images': n_images,
        'n_equations': n_equations,
        'n_text': n_text,
        'md_path': r['md_path'],
    })

try:
    import pandas as pd
    df = pd.DataFrame(records)
    display(df.drop(columns=['md_path']))
except Exception:
    df = None
    for rec in records:
        print(rec)

,pdf,ok,backend,latency_s,pages,chars,n_tables,n_images,n_equations,n_text
0,AF SE 2,True,marker,116.61,15,24134,8,9,0,20
1,Admin Eval Framework.docx,True,marker,22.23,7,10413,0,0,0,2
2,Agenda_ Operational Leadership Institute (SY 2...,True,marker,23.40,6,14714,6,5,0,14
3,Athlos 2024-2025 Observation Rubric Teacher.docx,True,marker,201.46,13,66252,12,1,0,14
4,"Atlanta, GA Code of Ordinances",True,marker,51.71,15,20564,0,0,0,189
5,Benefits_HR Specialist Evaluation Form Templat...,True,marker,36.04,10,21210,2,10,0,8
6,Benefits_HR Specialist Evaluation Form Template,True,marker,36.32,10,21210,2,10,0,8
7,Bullseye Math Checklist - Sheet1,True,marker,1034.27,15,9027,14,0,0,0
8,Classroom Environment Checklist 6-8.docx,True,marker,26.52,5,17412,4,3,0,0
9,Marzano Rubrics with scales. potential evidenc...,True,marker,39.07,10,104449,17,0,0,38


## 7. Markdown → standalone HTML

Render each `.md` to a full HTML page using the **same house CSS** as the other notebooks, with
MathJax so Marker's LaTeX equations display, and rewrite image paths so extracted figures load.

In [ ]:
import markdown as md_lib

HOUSE_CSS = (
    'body{font-family:Arial,Helvetica,sans-serif;max-width:900px;margin:24px auto;'
    'padding:0 16px;line-height:1.5}'
    'table{border-collapse:collapse;margin:12px 0}'
    'td,th{border:1px solid #888;padding:4px 8px;vertical-align:top}'
    'h1,h2,h3{margin:.6em 0 .3em}img{max-width:100%}'
)
MATHJAX = (
    "<script>window.MathJax={tex:{inlineMath:[['$','$'],['\\\\(','\\\\)']],"
    "displayMath:[['$$','$$'],['\\\\[','\\\\]']]}};</script>"
    '<script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js"></script>'
)

def to_html(md_path: Path, stem: str) -> Path:
    raw  = md_path.read_text(encoding='utf-8')
    body = md_lib.markdown(raw, extensions=['tables', 'fenced_code'])
    base = md_path.parent.resolve().as_uri()
    body = body.replace('src="images/', f'src="{base}/images/')
    html = (f'<!doctype html><html><head><meta charset="utf-8">'
            f'<title>{stem}</title><style>{HOUSE_CSS}</style>{MATHJAX}'
            f'</head><body>{body}</body></html>')
    out = HTML_DIR / f'{stem}.html'
    out.write_text(html, encoding='utf-8')
    return out

for rec in records:
    if rec['md_path']:
        to_html(Path(rec['md_path']), rec['pdf'])
print('wrote', len(list(HTML_DIR.glob('*.html'))), 'HTML pages to', HTML_DIR)

wrote 10 HTML pages to /Users/asikifthakerhamim/Documents/ocr_exploration/notebooks/marker/html


: 

## 8. Preview a result inline

Change `PREVIEW` to any PDF stem from the table above.

In [ ]:
from IPython.display import HTML, display

PREVIEW = records[0]['pdf'] if records else None
page = HTML_DIR / f'{PREVIEW}.html'
if PREVIEW and page.exists():
    print(page)
    display(HTML(page.read_text(encoding='utf-8')))
else:
    print('nothing to preview')